# CS336 Spring 2026 - 08 BPE Tokenizer Inference: Encode / Decode / Merge Rank
----
> 07 完成了训练前的语料统计；05 产出了 `vocab` 和有序 `merges`。本节把它们变成真正可用的 Tokenizer。

```text
text → isolate special tokens → GPT-style pre-tokens
     → UTF-8 bytes → apply learned BPE merges by rank → token IDs
token IDs → vocab lookup → concatenate bytes → UTF-8 decode
```

这里讨论的是**推理编码**，不再统计训练语料中的 pair frequency。

## 1. Merge rank 是什么？

训练时，我们按语料里的频率反复选择 winner，得到**有序**列表：

```text
rank 0: (b"h", b"e")     => b"he"
rank 1: (b"he", b"l")    => b"hel"
rank 2: (b"hel", b"l")   => b"hell"
rank 3: (b"hell", b"o")  => b"hello"
```

推理时见到多个候选相邻 pair，优先使用**训练时排名靠前(rank 更小)** 的 pair，而不是重新计算其在输入中的频率。


In [ ]:
from __future__ import annotations

import regex

GPT2_PRETOKEN_PATTERN = (
    r"'(?:[sdmt]|ll|ve|re)"
    r"| ?\p{L}+"
    r"| ?\p{N}+"
    r"| ?[^\s\p{L}\p{N}]+"
    r"|\s+(?!\S)"
    r"|\s+"
)


def pretokenize(text: str) -> list[str]:
    return [m.group(0) for m in regex.finditer(GPT2_PRETOKEN_PATTERN, text)]


MERGES = [
    (b"h", b"e"),
    (b"he", b"l"),
    (b"hel", b"l"),
    (b"hell", b"o"),
    (b" ", b"w"),
    (b" w", b"o"),
    (b" wo", b"r"),
    (b" wor", b"l"),
    (b" worl", b"d"),
]
MERGE_RANKS = {pair: rank for rank, pair in enumerate(MERGES)}
print("Merge ranks:", MERGE_RANKS)

Merge ranks: {(b'h', b'e'): 0, (b'he', b'l'): 1, (b'hel', b'l'): 2, (b'hell', b'o'): 3, (b' ', b'w'): 4, (b' w', b'o'): 5, (b' wo', b'r'): 6, (b' wor', b'l'): 7, (b' worl', b'd'): 8}


## 2. Encode 单个 pre-token

输入 `"hello"` 先变成 `[b"h", b"e", b"l", b"l", b"o"]`。

每轮找到当前相邻 pair 中 rank 最小的一个；之后左到右将所有不重叠的该 pair 合并。这个教学版本直接扫描列表，容易观察，但不是高性能实现。

**重要不变量：BPE merge 不跨 pre-token boundary。** `"hello world"` 预分词得到的 `"hello"` 与 `" world"` 要分开编码。


In [2]:
def merge_pair_once(tokens: list[bytes], pair: tuple[bytes, bytes]) -> list[bytes]:
    out = []
    i = 0

    while i < len(tokens):
        if i + 1 < len(tokens) and (tokens[i], tokens[i + 1]) == pair:
            out.append(tokens[i] + tokens[i + 1])
            i += 2
        else:
            out.append(tokens[i])
            i += 1
    return out


def encode_piece_as_bytes(piece: str, ranks: dict) -> list[bytes]:
    tokens = [bytes([b]) for b in piece.encode("utf-8")]
    while len(tokens) > 1:
        candidates = [pair for pair in zip(tokens, tokens[1:]) if pair in ranks]
        if not candidates:
            break
        winner = min(candidates, key=ranks.__getitem__)
        tokens = merge_pair_once(tokens, winner)
    return tokens


for piece in pretokenize("hello world!"):
    print(repr(piece), "→", encode_piece_as_bytes(piece, MERGE_RANKS))

assert encode_piece_as_bytes("hello", MERGE_RANKS) == [b"hello"]
assert encode_piece_as_bytes(" world", MERGE_RANKS) == [b" world"]
assert encode_piece_as_bytes("hello world", MERGE_RANKS) != [b"hello world"]

'hello' → [b'hello']
' world' → [b' world']
'!' → [b'!']


## 3. 从 256 个基础 bytes 建立 vocab

每个 UTF-8 byte 都是一个基础 token：`id = 0,...,255`。每条 merge 追加一个新 token，其 bytes 是两个子 token bytes 的拼接。

Special token 也需要独立 ID，其文本 bytes 原样保存，但不能由普通 BPE 算法“碰巧拼出来”。

实际 Assignment 的接口由官方仓库定义；这里使用教学版接口解释不变量。


In [3]:
def build_vocab(merges, special_tokens):
    vocab = {i: bytes([i]) for i in range(256)}
    for left, right in merges:
        vocab[len(vocab)] = left + right
    for token in special_tokens:
        vocab[len(vocab)] = token.encode("utf-8")
    return vocab


SPECIAL_TOKENS = ["<|endoftext|>", "<|pad|>"]
vocab = build_vocab(MERGES, SPECIAL_TOKENS)
bytes_to_id = {token_bytes: token_id for token_id, token_bytes in vocab.items()}
special_to_id = {s: bytes_to_id[s.encode("utf-8")] for s in SPECIAL_TOKENS}

assert len(vocab) == len(set(vocab.values()))
assert all(vocab[i] == bytes([i]) for i in range(256))

print("vocab size:", len(vocab))
print("special IDs:", special_to_id)
print("hello ID:", bytes_to_id[b"hello"])

vocab size: 267
special IDs: {'<|endoftext|>': 265, '<|pad|>': 266}
hello ID: 259


## 4. Special Token 检测先于普通 regex

延续 04 的原则：先使用完整 special-token 字符串建立 pattern；若某个 special token 是另一个的前缀，按**长度递减**排列，避免短 token 抢先匹配。

匹配出来的特殊片段直接输出其独立 ID；其间的 ordinary spans 才进入 pre-tokenization 与 BPE。

In [4]:
def compile_special_pattern(special_tokens):
    if not special_tokens:
        return None
    ordered = sorted(set(special_tokens), key=lambda s: (-len(s), s))
    return regex.compile("|".join(regex.escape(s) for s in ordered))


class TeachingBPETokenizer:
    def __init__(self, vocab, merges, special_tokens):
        self.vocab = dict(vocab)
        self.reverse_vocab = {v: k for k, v in self.vocab.items()}
        self.merge_ranks = {pair: i for i, pair in enumerate(merges)}
        self.special_to_id = {
            s: self.reverse_vocab[s.encode("utf-8")] for s in special_tokens
        }
        self.special_pattern = compile_special_pattern(special_tokens)

    def _encode_ordinary(self, text):
        result = []
        for piece in pretokenize(text):
            for token_bytes in encode_piece_as_bytes(piece, self.merge_ranks):
                result.append(self.reverse_vocab[token_bytes])
        return result

    def encode(self, text):
        if self.special_pattern is None:
            return self._encode_ordinary(text)
        result = []
        cursor = 0
        for match in self.special_pattern.finditer(text):
            result.extend(self._encode_ordinary(text[cursor : match.start()]))
            result.append(self.special_to_id[match.group(0)])
            cursor = match.end()
        result.extend(self._encode_ordinary(text[cursor:]))
        return result

    def decode(self, token_ids, errors="replace"):
        # 一定要先拼完整 bytes，最后统一 UTF-8 decode。
        try:
            data = b"".join(self.vocab[int(i)] for i in token_ids)
        except KeyError as err:
            raise ValueError(f"unknown token id: {err}") from err
        return data.decode("utf-8", errors=errors)


tokenizer = TeachingBPETokenizer(vocab, MERGES, SPECIAL_TOKENS)
text = "hello world!<|endoftext|>你🌍<|pad|>"
ids = tokenizer.encode(text)
print("encoded ids:", ids)
print("decoded:", tokenizer.decode(ids))
assert tokenizer.decode(ids) == text
assert special_to_id["<|endoftext|>"] in ids

encoded ids: [259, 264, 33, 265, 228, 189, 160, 240, 159, 140, 141, 266]
decoded: hello world!<|endoftext|>你🌍<|pad|>


## 5. Decode 的关键：先连接 bytes，再按 UTF-8 解码

单个 token 不一定是完整 Unicode 字符。例如中文 `"你"` 对应三个 UTF-8 bytes，若逐 token `.decode()`，每个 byte 都可能单独报错。

正确的 decode 是：

$$
\mathrm{decode}(i_1,\dots,i_n)
=\mathrm{UTF8Decode}\bigl(vocab[i_1] \Vert \cdots \Vert vocab[i_n]\bigr).
$$

默认 `errors="replace"` 允许不完整/损坏的字节流转为替换字符；对于正常 `str` 编码得到的 ID 序列，应保证严格 round-trip。

In [5]:
chinese = "你"
char_bytes = chinese.encode("utf-8")
print("single-character bytes:", list(char_bytes))
try:
    bytes([char_bytes[0]]).decode("utf-8")
except UnicodeDecodeError:
    print("逐 token 解码失败（符合预期）")

byte_token_ids = list(char_bytes)
assert tokenizer.decode(byte_token_ids, errors="strict") == chinese
print("先拼 bytes 再 decode：通过")

single-character bytes: [228, 189, 160]
逐 token 解码失败（符合预期）
先拼 bytes 再 decode：通过


## 6. Round-trip、Special Token、边界情况

验证至少覆盖空字符串、ASCII、中文、emoji、换行、连续 special tokens、相邻的特殊与普通文本，以及不存在任何已训练 merge 的片段。

注意 `decode(encode(text)) == text` 是必要条件，但**不能单独证明 BPE 合并顺序正确**：即便不合并任何 bytes，也能 round-trip。因此还需断言关键输入的 token IDs 或 token bytes 与预期一致。


In [6]:
samples = [
    "",
    "hello",
    "hello world",
    "hello!!\n",
    "你好，世界🌍",
    "a\tb\nc",
    "🌍🌍🌍",
    "<|endoftext|>",
    "<|endoftext|><|endoftext|>",
    "hello<|endoftext|>world",
    "<|pad|>你好<|endoftext|>",
]
for s in samples:
    encoded = tokenizer.encode(s)
    assert tokenizer.decode(encoded, errors="strict") == s, repr(s)
assert tokenizer.encode("hello") == [bytes_to_id[b"hello"]]
assert tokenizer.encode(" world") == [bytes_to_id[b" world"]]
assert tokenizer.encode("<|endoftext|>") == [special_to_id["<|endoftext|>"]]
assert tokenizer.encode("") == []
print("11 组 round-trip + merge rank + special-token 不变量：通过")


11 组 round-trip + merge rank + special-token 不变量：通过


## 7. Chunked encode 与 streaming encode 不同

“文件安全切块”服务于**训练阶段的频率统计**。对于 encode 流，随意把一个 pre-token 拆成两部分会改变编码 ID 序列；即使 decode 后文本仍相同，tokenization 也不等价。

正式的 `encode_iterable` 必须清楚其语义：是分别编码独立文档，还是无损地把任意文本分段组合成一个连续文档。后者需要维护跨片段缓冲或明确的语义边界。


In [ ]:
whole = tokenizer.encode("hello")
parts = tokenizer.encode("he") + tokenizer.encode("llo")
print("whole ids:", whole)
print("naively streamed ids:", parts)
assert whole != parts
assert tokenizer.decode(parts) == "hello"
print("decode 一样，不代表 token IDs 一样：已验证")

whole ids: [259]
naively streamed ids: [256, 108, 108, 111]
decode 一样，不代表 token IDs 一样：已验证


## 8. 算法复杂度与真实工程

教学实现每轮扫描所有相邻 pairs，适合短字符串和 trace。大规模 encode 通常会优化局部 pair 更新，避免重复构造中间列表，并保证 deterministic behavior。

训练阶段交付的核心文件形式可能是 `vocab`、`merges` 与 `special_tokens`；实际序列化格式应以官方 Assignment 要求为准，不把这个教学类当作官方参考实现。


## 9. 本章总结 → 09 Tensor Shapes / Linear / Embedding

Tokenizer 的输出是整数 token ID 序列，其后进入 Neural Network：

```text
text → Tokenizer → token ids (B,T)
     → Embedding → hidden states (B,T,D)
     → Transformer → logits (B,T,V)
```

接下来开始追踪张量 shape、参数矩阵和梯度。
